# 03. 오픈 LLM — 모델 카드 코드 그대로 띄우기

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/acho98/gunyang-ai-api/blob/main/labs/03_open_llm_modelcard.ipynb)

> 🧭 **스토리** — 견적을 낸 당신에게 팀장이 묻는다. "API 말고 우리 서버에 모델을 직접 띄우면?" 먼저 공개 모델 하나를 **모델 카드에 적힌 코드 그대로** 띄우고, 모델이 실제로 받는 문자열을 들여다본다.
>
> 📖 **이론 연결** — 이론1 p37~57 (다음 토큰 예측·채팅 템플릿·비결정성), p124~125 (특수 토큰), p83~86 (디코딩·temperature·max_tokens) · ⏱ **13분** · 🖥 **GPU 필수**
>
> **런타임 → 런타임 유형 변경 → T4 GPU** 를 먼저 선택하세요. (GPU가 없으면 작은 설정으로 느리게 돌아갑니다.)
> 💡 **앞 노트북을 열어 둔 상태라면** 그 노트북에서 `런타임 → 런타임 연결 해제 및 삭제`를 먼저 하세요. 무료 코랩은 GPU 세션을 하나만 허용합니다.

**셀 배지**: ⚙️ 준비 · 🔵 따라 실행 · 🟡 바꿔서 실행 · 🤔 생각해보기 · 🟠 심화(시간 남을 때)

## 0. 준비 ⚙️
모델 카드가 요구하는 `transformers>=4.54`와 `accelerate`만 설치합니다. 모델은 **LG AI연구원 EXAONE 4.0 1.2B**(한국어 공식 지원, 연구·교육용 라이선스)입니다.
GPU가 없으면 자동으로 더 작은 `Qwen/Qwen2.5-0.5B-Instruct`로 바꾸고 생성 길이를 40토큰으로 줄입니다.

In [ ]:
%pip -q install -U "transformers>=4.54" accelerate

In [ ]:
# ⚙️ GPU 확인 — T4가 잡혔는지, 메모리가 얼마인지
import torch, subprocess
HAS_GPU = torch.cuda.is_available()
if HAS_GPU:
    name = torch.cuda.get_device_name(0)
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {name} · 메모리 {total_gb:.1f} GB")
else:
    print("GPU 없음 — CPU로 진행합니다(느립니다). 런타임 유형에서 T4를 선택하면 빨라집니다.")
DEVICE = "cuda" if HAS_GPU else "cpu"
DTYPE = torch.float16 if HAS_GPU else torch.float32   # T4는 bf16을 지원하지 않아 fp16을 씁니다

## 1. 모델 카드 퀵스타트 그대로 로드 🔵
[EXAONE 4.0 1.2B 모델 카드](https://huggingface.co/LGAI-EXAONE/EXAONE-4.0-1.2B)의 코드와 같습니다. `trust_remote_code`가 필요 없습니다(transformers 4.54부터 EXAONE 4.0 내장).
T4에서는 다운로드(2.6GB)와 로드에 1분 안팎 걸립니다.

In [ ]:
import os, time
from transformers import AutoModelForCausalLM, AutoTokenizer

SMOKE = os.environ.get("NB_SMOKE") == "1"          # 강사 검증용 스위치 — 수강생은 신경 쓰지 않아도 됩니다
if HAS_GPU and not SMOKE:
    MODEL_NAME, MAX_NEW = "LGAI-EXAONE/EXAONE-4.0-1.2B", 150
else:                                             # CPU 폴백: 더 작은 모델 · 짧은 생성
    MODEL_NAME, MAX_NEW = "Qwen/Qwen2.5-0.5B-Instruct", 40

t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=DTYPE,                                  # 카드 표기 torch_dtype 의 새 이름. GPU=fp16, CPU=fp32
    device_map="auto" if HAS_GPU else None,
)
n_params = sum(p.numel() for p in model.parameters())
print(f"모델: {MODEL_NAME} · 파라미터 {n_params/1e9:.2f}B · dtype {model.dtype} · 로드 {time.time()-t0:.0f}초")
print(f"가중치 메모리(get_memory_footprint): {model.get_memory_footprint()/1e9:.2f} GB")
if HAS_GPU:
    print(f"GPU 메모리 사용(memory_allocated): {torch.cuda.memory_allocated()/1e9:.2f} GB")

💡 **해석** — 파라미터 수 × 바이트 수 ≈ 메모리입니다. EXAONE 4.0 1.2B의 실제 파라미터는 1.28B이고, fp16(2바이트)으로 올리면 약 2.56GB입니다(강사 환경 CPU fp32에서는 5.12GB로 정확히 두 배). T4 메모리(약 15GB)의 6분의 1 정도라서 여유가 있습니다. CPU 폴백(fp32, 4바이트)은 같은 파라미터라도 메모리가 두 배입니다. 노트북 04에서 이 숫자를 4비트로 줄여 봅니다.

## 2. 모델이 실제로 받는 문자열 — 채팅 템플릿 🔵
우리는 `messages` 리스트(역할 + 내용)를 넘기지만, 모델이 받는 것은 **특수 토큰이 섞인 문자열 하나**입니다. `apply_chat_template(..., tokenize=False)`로 그 문자열을 그대로 찍어 봅니다.

In [ ]:
messages = [{"role": "user", "content": "한빛테크 신입 사원에게 출장비 정산 요령을 한 문장으로 알려줘."}]

prompt_str = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print("=== add_generation_prompt=True (생성할 때) ===")
print(repr(prompt_str))

no_gen = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
print("\n=== add_generation_prompt=False (대화 기록 저장·학습용) ===")
print(repr(no_gen))
print("\n차이 나는 꼬리:", repr(prompt_str[len(no_gen):]))

💡 **해석** — EXAONE 4.0은 `[|user|]` · `[|endofturn|]` · `[|assistant|]` 같은 **특수 토큰**으로 역할을 구분합니다(이론1 p124~126). 강사 환경에서 찍은 실제 문자열은 이렇습니다.
```
'[|user|]\n한빛테크 … 알려줘.[|endofturn|]\n[|assistant|]\n<think>\n\n</think>\n\n'
```
- `add_generation_prompt=True`이면 끝에 `[|assistant|]\n`이 붙습니다. "이제 네 차례"라는 신호입니다. 이게 없으면 모델이 사용자 말을 이어 쓰기도 합니다.
- 끝의 빈 `<think>\n\n</think>`는 **사고 모드를 끈 상태**라는 표시입니다(6절에서 켭니다).
- CPU 폴백인 Qwen2.5는 `<|im_start|>` · `<|im_end|>`를 쓰고, 시스템 메시지를 주지 않으면 기본 문장 "You are Qwen…"을 끼워 넣습니다. **모델마다 템플릿이 다르므로 문자열을 직접 만들지 말고 `apply_chat_template`을 쓰세요.**

## 3. 생성 — greedy vs 샘플링 🔵
같은 질문을 **greedy**(매번 확률 1등 토큰)로 한 번, **샘플링**(`temperature=0.7, top_p=0.9`)으로 세 번 생성합니다. 아래 `generate()`는 이후 모든 절에서 다시 씁니다.

In [ ]:
import torch

def generate(messages, max_new_tokens=MAX_NEW, enable_thinking=False, **gen_kw):
    """채팅 템플릿 → 토큰 → 생성 → 새로 만든 부분만 디코딩. (답, 생성 토큰 수, 초) 반환"""
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True,
        enable_thinking=enable_thinking,          # EXAONE·Qwen3 템플릿 옵션. 지원하지 않는 템플릿은 무시합니다
    ).to(model.device)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, **gen_kw)
    new_ids = out[0, inputs["input_ids"].shape[1]:]          # 프롬프트 뒤에 새로 붙은 토큰만
    return tokenizer.decode(new_ids, skip_special_tokens=True).strip(), len(new_ids), time.time() - t0

q = [{"role": "user", "content": "출장 보고서를 쓸 때 꼭 들어가야 할 항목 세 가지는?"}]
ans, n, sec = generate(q, do_sample=False)
print(f"[greedy] {n}토큰 · {sec:.1f}초\n{ans}\n")
for i in range(3):
    ans, n, sec = generate(q, do_sample=True, temperature=0.7, top_p=0.9)
    print(f"[샘플링 {i+1}] {n}토큰 · {sec:.1f}초\n{ans}\n")

💡 **해석** — greedy는 다시 돌려도 같은 답이 나옵니다. 샘플링 세 번은 **문장과 항목이 서로 다릅니다**. 강사 환경 EXAONE greedy 답은 "목적 및 배경 정보 → 일정 및 계획 대비 진행 상황 → …" 순서였고 120~150토큰 상한에서는 셋째 항목 전에 잘리기도 합니다(4절). 모델은 "다음 토큰의 확률 분포"를 내놓을 뿐이고, 그 분포에서 어떻게 고르느냐(디코딩 전략)가 답을 바꿉니다(이론1 p37, p83~85). 규정 안내처럼 **일관성이 중요한 업무는 greedy나 낮은 temperature**, 아이디어 내기처럼 다양성이 필요한 업무는 샘플링이 맞습니다.

## 4. `max_new_tokens` — 답이 잘리는 이유 🔵
생성 길이 상한을 아주 작게 주면 모델은 문장 중간에서 멈춥니다. API의 `max_output_tokens`와 같은 개념입니다.

In [ ]:
for limit in [10, MAX_NEW]:
    ans, n, sec = generate(q, max_new_tokens=limit, do_sample=False)
    cut = "← 상한에 걸려 잘림" if n >= limit else "← 모델이 스스로 끝냄(EOS)"
    print(f"max_new_tokens={limit:>3} → {n}토큰 {cut}\n  {ans}\n")

💡 **해석** — 생성 토큰 수가 상한과 같으면 잘린 것입니다. 상한보다 적으면 모델이 끝 토큰(EOS)을 내고 스스로 멈춘 것입니다. 서비스에서 답이 뚝 끊기면 먼저 이 값을 확인하세요. 반대로 상한은 **출력 비용의 안전장치**이기도 합니다(노트북 02의 출력 단가 5배).

## 5. 스트리밍 — 한 토큰씩 보여 주기 🔵
챗봇 화면에서 글자가 흘러나오는 것은 생성이 끝나기를 기다리지 않고 **토큰이 나오는 대로** 화면에 보내기 때문입니다. `TextIteratorStreamer` + 스레드로 구현합니다.

In [ ]:
from threading import Thread
from transformers import TextIteratorStreamer

inputs = tokenizer.apply_chat_template(q, add_generation_prompt=True, return_tensors="pt",
                                       return_dict=True, enable_thinking=False).to(model.device)
streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
th = Thread(target=model.generate, kwargs=dict(**inputs, streamer=streamer, max_new_tokens=MAX_NEW, do_sample=False))
t0 = time.time(); th.start()
first = None
for piece in streamer:                      # 생성 스레드가 토큰을 만들 때마다 조각이 도착
    if first is None and piece.strip():
        first = time.time() - t0
    print(piece, end="", flush=True)
th.join()
print(f"\n\n첫 조각까지 {first or 0:.2f}초 · 전체 {time.time()-t0:.1f}초")

💡 **해석** — 전체 시간은 3절과 비슷하지만 **첫 조각이 훨씬 빨리** 보입니다. 사용자가 느끼는 속도는 전체 시간보다 "첫 토큰까지 시간(TTFT)"이 좌우합니다. OpenAI API의 `stream=True`도 같은 원리입니다.

## 6. 사고(thinking) 모드 — 생각을 길게 쓰면 정답률이 오를까? 🔵
EXAONE 4.0은 하나의 모델에 **사고 모드**와 **비사고 모드**가 함께 있습니다. 템플릿에 `enable_thinking=True`를 주면 모델이 `<think> … </think>` 안에 풀이 과정을 먼저 쓰고 답합니다. 한빛테크 규정으로 계산 문제를 내 봅니다(정답 **395,000원** = 3일 × (일비 30,000 + 식비 35,000) + 2박 × 100,000).
카드 권장값대로 사고 모드는 `temperature=0.6, top_p=0.95` 샘플링을 씁니다.

In [ ]:
calc = [{"role": "user", "content":
         "과장이 부산으로 2박 3일 출장을 간다. 일비는 하루 30,000원, 식비는 하루 35,000원이고 "
         "숙박비는 1박에 100,000원을 썼다. 총 출장비는 얼마인가? 마지막 줄에 금액만 써라."}]
THINK_MAX = 1536 if MAX_NEW > 40 else 40          # 사고 모드는 1,000토큰 안팎을 씁니다. CPU 폴백은 짧게 자릅니다

for think_on in [False, True]:
    kw = dict(do_sample=True, temperature=0.6, top_p=0.95) if think_on else dict(do_sample=False)
    ans, n, sec = generate(calc, max_new_tokens=THINK_MAX if think_on else MAX_NEW, enable_thinking=think_on, **kw)
    if "</think>" in ans:                          # 사고 과정과 최종 답을 나눠서 봅니다
        thought, final = ans.split("</think>", 1)
        print(f"[사고 모드 ON] 총 {n}토큰 · {sec:.1f}초 · 사고 부분 {len(tokenizer.encode(thought))}토큰")
        print("  최종 답:", final.strip()[-200:], "\n")
    else:
        note = " (상한 안에 사고가 끝나지 않음)" if think_on and MODEL_NAME.startswith("LGAI") else ""
        print(f"[사고 모드 {'ON' if think_on else 'OFF'}] {n}토큰 · {sec:.1f}초{note}")
        print("  답:", ans[-300:], "\n")
if not MODEL_NAME.startswith("LGAI"):
    print("(안내) CPU 폴백 모델(Qwen2.5)은 사고 모드가 없어 두 결과가 비슷합니다. T4에서 EXAONE으로 다시 보세요.")

💡 **해석** — 강사 환경에서 EXAONE 1.2B로 같은 문제를 풀린 결과입니다.
- **비사고 모드**: 바로 답을 써서 **295,000원**(틀림). 숙박비를 2박이 아니라 1박만 더했습니다.
- **사고 모드**: 풀이에 **879~1,147토큰**을 썼습니다(비사고 답의 약 10배). 한 번은 **395,000원**(정답)을 냈고, 다른 한 번은 "2박 3일"을 5일로 잘못 읽어 825,000원을 냈습니다.

사고 모드는 정답률을 **올려 주지만 보장하지는 않습니다**. 소형 모델일수록 그렇습니다. 그리고 사고 토큰도 **출력 토큰으로 과금**되고, T4에서도 수십 초가 걸립니다. API의 `reasoning.effort`가 바로 이 사고 길이를 조절하는 손잡이입니다(노트북 12).

## 7. 툴(함수) 스키마는 템플릿 안에서 어떻게 보일까 🔵 — 노트북 13 예고
파이썬 함수에 **구글 스타일 docstring**을 달아 `tools=[함수]`로 넘기면, transformers가 JSON 스키마로 바꿔 템플릿에 끼워 넣습니다. 여기서는 **실행하지 않고 문자열만** 봅니다.

In [ ]:
def get_travel_allowance(grade: str, region: str) -> dict:
    """직급과 출장 지역에 맞는 일비·숙박비 상한을 규정집에서 찾아 돌려준다.

    Args:
        grade: 직급 (사원, 대리, 과장, 차장, 부장, 임원 중 하나)
        region: 출장 지역 (국내, 북미, 유럽, 일본·중국, 동남아시아, 기타 지역)
    """
    return {}

tool_prompt = tokenizer.apply_chat_template(
    [{"role": "user", "content": "과장이 유럽 출장 가면 숙박비 상한이 얼마야?"}],
    tools=[get_travel_allowance], tokenize=False, add_generation_prompt=True)
print(tool_prompt)

💡 **해석** — 함수 이름·설명·인자 설명이 **JSON 스키마가 되어 시스템 영역에 그대로 들어갑니다**. EXAONE은 `<tool>…</tool>`, Qwen은 `<tools>…</tools>` 태그로 감싸고, "호출하려면 `<tool_call>{name, arguments}</tool_call>` 형식으로 답하라"는 지시도 붙습니다. 모델은 함수를 실행하지 못합니다. **호출 요청 문자열을 쓸 뿐이고, 실행은 우리 코드가 합니다.** docstring을 잘 써야 모델이 툴을 제대로 고르는 이유가 여기 있습니다. 노트북 13에서 실제로 실행합니다.

## 8. 바꿔서 실행 🟡
1. **temperature**: `TEMPS`의 두 값(0.1, 1.5)으로 같은 한국어 질문을 생성해 비교하세요. EXAONE 카드는 한국어 답에서 영어·한자가 섞이는 현상(code-switching)을 줄이려면 **0.1 정도의 낮은 값**을 권합니다. 1.5에서는 어떤 일이 생기나요?
2. **사고 모드**: `MY_CALC`를 다른 산수 문제로 바꾸고 `THINK`를 `True`/`False`로 바꿔 토큰 수와 정답을 비교하세요.

In [ ]:
TEMPS = [0.1, 1.5]                                   # ← 바꿔 보세요 (예: 0.7, 2.0)
korean_q = [{"role": "user", "content": "한빛테크 신입 사원에게 보내는 환영 인사를 두 문장으로 써 줘."}]
for t in TEMPS:
    ans, n, sec = generate(korean_q, do_sample=True, temperature=t, top_p=1.0)
    print(f"[temperature={t}] {n}토큰\n{ans}\n")

💡 **해석** — 강사 환경 EXAONE에서 0.1은 "한빛테크 신입 사원님, 환영합니다! …"처럼 자연스러운 한국어였고, 1.5는 "오늘날 따뜻한Starting Point를 보내니 …✨"처럼 **영어가 섞이고 문장이 어색해졌습니다**. temperature가 높으면 확률이 낮은 토큰(다른 언어·이모지)도 뽑히기 때문입니다.

In [ ]:
MY_CALC = "사원이 당일치기로 왕복 150km 출장을 갔다. 일비가 하루 20,000원이고 당일 출장은 일비의 50%만 준다. 일비는 얼마인가?"   # ← 바꿔 보세요
THINK = True                                                                       # ← True / False
kw = dict(do_sample=True, temperature=0.6, top_p=0.95) if THINK else dict(do_sample=False)
ans, n, sec = generate([{"role": "user", "content": MY_CALC}], max_new_tokens=THINK_MAX if THINK else MAX_NEW,
                       enable_thinking=THINK, **kw)
print(f"사고 모드 {THINK} · {n}토큰 · {sec:.1f}초 (정답: 10,000원)\n{ans[-400:]}")

### 🤔 생각해보기

**사고 모드는 왜 비싸고 느린가?** 한빛테크 규정 도우미에서는 어떤 질문에만 사고 모드를 켜는 게 좋을까요?

<details><summary><b>🤔 생각해보기 — 해설 보기(클릭)</b></summary>

- **비싸고 느린 이유**: 풀이 과정도 한 토큰씩 **순차 생성**되는 출력 토큰입니다. 답이 20토큰이어도 사고가 500토큰이면 출력이 25배가 되고, 시간과 비용도 그만큼 늘어납니다(이론1 p65).
- **켤 때**: 여러 숫자를 곱하고 더하는 출장비 계산, 조건이 겹치는 규정 해석(직급 × 지역 × 일수), 여러 조항을 비교해야 하는 질문.
- **끌 때**: "정산 기한이 며칠이야?"처럼 문서에서 한 줄 찾아 답하면 되는 질문, 인사말·요약·분류처럼 추론이 필요 없는 작업.
- 실무에서는 질문을 먼저 분류하고 어려운 질문만 사고 모드(또는 API의 높은 `reasoning.effort`)로 보내는 **라우팅**을 씁니다.

</details>

---
### ✅ 이 노트북에서 가져갈 것
- 모델 카드의 퀵스타트 코드가 곧 공식 사용법이다. dtype·device_map만 환경에 맞춘다(T4는 fp16).
- 모델이 받는 것은 messages가 아니라 특수 토큰이 섞인 문자열이다. 템플릿은 `apply_chat_template`에 맡긴다.
- 디코딩(greedy·샘플링·temperature)이 답을 바꾸고, `max_new_tokens`가 답을 자른다.
- 사고 모드는 정확도를 사는 대신 토큰·시간을 쓴다. 툴 스키마도 결국 템플릿 속 문자열이다.
- 다음 노트북으로 가기 전 `런타임 → 런타임 연결 해제 및 삭제`로 GPU를 비우세요.

> 다음 노트북: **04_quantization — 같은 모델을 4분의 1 크기로**